# World Bank Business Opportunities — Consultant Services

Fetches live "Current Opportunities" (Consultant Services) from the World Bank's procurement
notices API, filters to English-language notices, and uploads new contracts to the unified
Notion database.

**Link:** https://projects.worldbank.org/en/projects-operations/opportunities

**Filters applied:**
- Procurement type: Consultant Services only (`procurement_group_desc_exact`)
- Sector: include-list of ~52 sectors, matching all sectors except the 6 excluded by Javiera
  (Health Facilities and Construction, Housing Construction, ICT Infrastructure, Irrigation
  and Drainage, Other Water Supply/Sanitation/Waste Management, Waste Management)
- Deadline: currently-open opportunities only (`deadline_strdate` = today)
- Language: English only, checked against the API's own `notice_lang_name` field

No CPV codes apply to this source (the World Bank uses its own sector taxonomy, already
filtered server-side), so the `CPV Codes` Notion field is populated with "Not Applicable"
rather than left looking like a scrape gap.

In [1]:
import os

NOTION_TOKEN = os.environ["NOTION_TOKEN"]

DATABASE_ID = '334701e728cb8096a94cebc0985684a2'

headers_notion = {
    "Authorization": "Bearer " + NOTION_TOKEN,
    "Content-Type": "application/json",
    "Notion-Version": "2022-06-28",
}


In [2]:
import requests
import pandas as pd
import json
import html
import os
from datetime import date, datetime, timezone
from dateutil import parser as _dateparser

WB_BASE_URL = "https://search.worldbank.org/api/v2/procnotices"

# Defensive headers — this endpoint sits behind Cloudflare Bot Management. Confirmed working
# without these during testing, but sending them costs nothing and matches the real browser.
WB_REQUEST_HEADERS = {
    "Origin": "https://projects.worldbank.org",
    "Referer": "https://projects.worldbank.org/",
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
}

PROCUREMENT_GROUP = "Consultant Services"

# Sector INCLUDE list (your 6 exclusions already removed) — same list used in the OppsLink
# build, confirmed live with Javiera.
SECTOR_INCLUDE_LIST = [
    "(Historic)Water supply and sanitation adjustment",
    "Adult, Basic and Continuing Education",
    "Agricultural Extension, Research, and Other Support Activities",
    "Agricultural markets, commercialization and agri-business",
    "Aviation",
    "Banking Institutions",
    "Capital Markets",
    "Crops",
    "Central Government (Central Agencies)",
    "Early Childhood Education",
    "Energy Transmission and Distribution",
    "Fisheries",
    "Forestry",
    "Health",
    "ICT",
    "Insurance and Pension",
    "Law and Justice",
    "Livestock",
    "Mining",
    "Other Agriculture, Fishing and Forestry",
    "Public Administration - Education",
    "Public Administration - Agriculture, Fishing & Forestry",
    "Primary Education",
    "Other Transportation",
    "Ports/Waterways",
    "Other Information and Communications Technologies",
    "Other Public Administration",
    "Other Industry,  and ",
    "Other Energy and Extractives",
    "Other Education",
    "Public Administration - Energy and Extractives",
    "Public Administration - Financial Sector",
    "Public Administration - Health",
    "Public Administration - Industry, Trade and Services",
    "Water Supply",
    "Workforce Development and Vocational Education",
    "Urban Transport",
    "Tertiary Education",
    "Tourism",
    "Sub-National Government",
    "Social Protection",
    "Secondary Education",
    "Public Administration - Information and Communications Technologies",
    "Public Administration - Social Protection",
    "Public Administration - Transportation",
    "Public Administration - Water,  and Waste Management",
    "Renewable Energy Biomass",
    "Renewable Energy Geothermal",
    "Renewable Energy Hydro",
    "Renewable Energy Solar",
    "Renewable Energy Wind",
    "Rural and Inter-Urban Roads",
]


def clean_description(description):
    if not description:
        return "Not Disclosed"
    cleaned = html.unescape(description)
    cleaned = cleaned.replace("\r\n", " ").replace("\n", " ")
    return " ".join(cleaned.split()).strip()


def fetch_wb_consultant_notices(max_pages: int = 20, rows_per_page: int = 100):
    all_notices = []
    today = date.today().isoformat()

    for page in range(max_pages):
        params = {
            "format": "json",
            "fl": "id,notice_type,noticedate,notice_lang_name,notice_status,"
                  "submission_deadline_date,submission_deadline_time,project_ctry_name,"
                  "project_id,project_name,bid_reference_no,bid_description,"
                  "procurement_group,procurement_method_code,procurement_method_name,"
                  "procurement_major_sector_name,contact_address,contact_ctry_name,"
                  "contact_email,contact_name,contact_organization,contact_phone_no,"
                  "contact_web_url,submission_date,notice_text",
            "srt": "noticedate",
            "order": "desc",
            "apilang": "en",
            "rows": rows_per_page,
            "srce": "both",
            "os": page * rows_per_page,
            "procurement_group_desc_exact": PROCUREMENT_GROUP,
            "sector_exact": "^".join(SECTOR_INCLUDE_LIST),
            "deadline_strdate": today,
        }

        try:
            resp = requests.post(WB_BASE_URL, params=params, headers=WB_REQUEST_HEADERS, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"⚠️ Error fetching page {page}: {e}")
            break

        batch = data.get("procnotices", [])
        print(f"📄 Page {page + 1}: fetched {len(batch)} notices (total so far: {len(all_notices) + len(batch)})")

        if not batch:
            break

        all_notices.extend(batch)

        if len(batch) < rows_per_page:
            break

    print(f"✅ Total notices fetched: {len(all_notices)}")
    return all_notices


In [3]:
# 1) Load already-uploaded titles to avoid duplicates
csv_path = "world_bank_contract_titles.csv"
existing_titles = set()
if os.path.exists(csv_path):
    try:
        prev = pd.read_csv(csv_path)
        if "Title" in prev.columns:
            existing_titles = set(prev["Title"].dropna().astype(str).str.strip().str.lower())
    except Exception as e:
        print("Warning: could not read", csv_path, ":", e)

# 2) Fetch + filter (English-only) + parse
wb_notices = fetch_wb_consultant_notices()

extracted_data = []
skipped_non_english = 0

for notice in wb_notices:
    lang = (notice.get("notice_lang_name") or "").strip()
    if lang != "English":
        skipped_non_english += 1
        continue

    title = (notice.get("bid_description") or "").strip()
    if not title:
        continue

    if title.strip().lower() in existing_titles:
        continue

    notice_id = notice.get("id", "")
    # Confirmed working — manually verified this URL pattern resolves to the correct notice
    link = f"https://projects.worldbank.org/en/projects-operations/procurement-detail/{notice_id}"

    description = clean_description(notice.get("notice_text") or notice.get("bid_description", ""))
    client_name = (notice.get("contact_organization") or "Not Disclosed").strip() or "Not Disclosed"
    client_link = notice.get("contact_web_url") or ""
    location = (notice.get("project_ctry_name") or "Not Specified").strip() or "Not Specified"

    extracted_data.append({
        "closing_date": notice.get("submission_deadline_date"),
        "country": location,
        "client": client_name,
        "client_link": client_link,
        "link": link,
        "title": title,
        "description": description,
        "value": "Unavailable",  # not published at this notice stage
        "cpv_codes": "Not Applicable",  # World Bank uses its own sector taxonomy, not CPV
        "language": lang,
    })

print(f"✅ {len(extracted_data)} new contracts ready for Notion upload (skipped {skipped_non_english} non-English notices)")


📄 Page 1: fetched 100 notices (total so far: 100)


📄 Page 2: fetched 98 notices (total so far: 198)
✅ Total notices fetched: 198
✅ 121 new contracts ready for Notion upload (skipped 77 non-English notices)


### Upload to Notion

In [4]:
def create_page(properties: dict):
    url = "https://api.notion.com/v1/pages"
    payload = {"parent": {"database_id": DATABASE_ID}, "properties": properties}
    res = requests.post(url, headers=headers_notion, json=payload, timeout=60)
    if not res.ok:
        print("❌ Notion error:", res.status_code, res.text[:500])
    else:
        print(f"✅ Page created: {properties['Name']['title'][0]['text']['content']}")
    return res


def _safe_str(x, default=""):
    if x is None:
        return default
    s = str(x).strip()
    return s if s else default


def _safe_iso(dt_str):
    if not dt_str:
        return None
    try:
        parsed = _dateparser.parse(dt_str)
        return parsed.isoformat()
    except Exception:
        return None


now_iso = datetime.now(timezone.utc).isoformat()
new_titles_for_csv = []

# Upload newest first, matching the other Notion notebooks' convention
for contract in list(reversed(extracted_data)):
    name = _safe_str(contract.get("title"))[:1000]
    if not name:
        continue

    closing_date_iso = _safe_iso(contract.get("closing_date"))

    props = {
        "Name": {"title": [{"text": {"content": name}}]},
        "CPV Codes": {"rich_text": [{"text": {"content": _safe_str(contract.get("cpv_codes"))[:2000]}}]},
        "Client": {"rich_text": [{"text": {"content": _safe_str(contract.get("client"), "Not Disclosed")[:2000]}}]},
        "Contract Link": {"url": _safe_str(contract.get("link")) or None},
        "Date Added": {"date": {"start": now_iso, "end": None}},
        "Closing Date": {"date": {"start": closing_date_iso, "end": None}} if closing_date_iso else {"date": None},
        "Description": {"rich_text": [{"text": {"content": _safe_str(contract.get("description"), "Not Disclosed")[:2000]}}]},
        "Employer Website": {"url": _safe_str(contract.get("client_link")) or None},
        "Language": {"rich_text": [{"text": {"content": _safe_str(contract.get("language"))[:2000]}}]},
        "Location": {"rich_text": [{"text": {"content": _safe_str(contract.get("country"))[:2000]}}]},
        "Reviewed By": {"select": {"name": "N/A"}},
        "Review Status": {"select": {"name": "Not Reviewed"}},
        "Value": {"rich_text": [{"text": {"content": _safe_str(contract.get("value"), "Unavailable")[:2000]}}]},
        "Contract Status": {"select": {"name": "Open"}},
        "Source": {"select": {"name": "World Bank"}},
    }

    try:
        create_page(props)
        new_titles_for_csv.append({"Title": name})
    except Exception as e:
        print(f"Error creating Notion page for '{name}': {e}")

# Save newly uploaded titles to CSV for next run's dedup
if new_titles_for_csv:
    new_df = pd.DataFrame(new_titles_for_csv, columns=["Title"])
    header_needed = not os.path.exists(csv_path)
    new_df.to_csv(csv_path, mode="a", header=header_needed, index=False)

print(f"✅ Uploaded {len(new_titles_for_csv)} new World Bank contracts to Notion.")


✅ Page created: Supervision of the construction works of the infrastructure of ISCED Uíge


✅ Page created: Supervision of the Construction works of the Infrastructure of ISCED Huila


✅ Page created: External Financial Auditor


✅ Page created: Consulting Services for A Baseline Assessment and Contextual Analysis of Kakuma and Dadaab–Hagadera Municipalities


✅ Page created: Consultant Service for support PMD for managing the implementation of DHST and CCST


...

✅ Page created: Provision of independent third-party Environmental and Social Monitoring services during the construction phase


✅ Page created: International PR and Media Outreach Campaign for IT Park Uzbekistan


✅ Page created: Consulting Services for the Development of an Institutional Strengthening Strategy, Roadmap, and Action Plan for Sustainable Agricultural Land Management in Georgia


✅ Page created: Serviços de Consultoria para Verificação do Projecto e Supervisão das Obras de Construção das Infra-Estruturas, Acesso Rodoviário E Subestação E Linhas de Ligação da Plataforma Logística da Caála


✅ Page created: System Integrator (SI) for MoMs LMIS Architecture Revitalization, Development, Enhancement, and Providing Implementation Support Services
✅ Uploaded 121 new World Bank contracts to Notion.
